# Item-based Collaborative Filtering Pipeline

This notebook focuses on a **pure Item-based Collaborative Filtering (ItemCF)** solution for the Member C task.

Prerequisites:
- The MIND data is available under `../data/MINDsmall_train` and `../data/MINDsmall_dev`.
- `pandas`, `numpy`, `matplotlib`, and `scikit-learn` are available in the environment.

Objectives:
- Build item-item similarity from click co-occurrence in `behaviors.tsv`.
- Improve the original ItemCF baseline without changing the core method.
- Tune several ItemCF-only settings and keep the best-performing configuration.


## Model Introduction

This notebook keeps the solution fully inside the **ItemCF family**.

The idea is simple:
1. Build an item-item similarity graph from user click history.
2. For each impression, score candidate news by aggregating similarities from the user's clicked history.
3. Tune only ItemCF-related components such as similarity formula, top-K neighbors, history length, recency decay, and popularity penalty.

Input:
- User click history and labeled impressions from the MIND dataset.
- News metadata used only for analysis and optional reporting.

Output:
- A pure ItemCF ranking model.
- A tuning table for multiple ItemCF settings.
- Final evaluation metrics including AUC, GroupAUC, MRR, and nDCG.


## Outline
1. Setup and data paths
2. Load behavior logs and news metadata
3. Build the ItemCF similarity matrix
4. Define stronger ItemCF scoring settings
5. Generate ItemCF candidate scores
6. Evaluate and save ItemCF outputs
7. Tune several ItemCF configurations
8. Select and export the best ItemCF model
---


## 1. Setup and data paths
Get ready by importing necessary libraries and defining data paths.


In [16]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict
import math
from tqdm import tqdm

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)

#  Define data paths
BASE_PATH = "../data"

TRAIN_PATH = os.path.join(BASE_PATH, "MINDsmall_train")
VALID_PATH = os.path.join(BASE_PATH, "MINDsmall_dev")

OUTPUT_PATH = "outputs/shared"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Check if data paths exist
assert os.path.exists(TRAIN_PATH), f"❌ Train path not found: {TRAIN_PATH}"
assert os.path.exists(VALID_PATH), f"❌ Valid path not found: {VALID_PATH}"

print("✅ Train path:", TRAIN_PATH)
print("✅ Valid path:", VALID_PATH)

✅ Train path: ../data\MINDsmall_train
✅ Valid path: ../data\MINDsmall_dev


## 2. Load the data
Load the shared candidate tables and news metadata. The candidate tables contain user click histories, while the news metadata contains information about each news item.

Three key dataframes:
- `train_clicks`: User click history for the training set
- `valid_clicks`: User click history for the validation set 
- `news_meta`: Metadata for news items (e.g., category, subcategory, title)


In [17]:
# Load behaviors.tsv

def load_behaviors(path):
    file_path = os.path.join(path, "behaviors.tsv")
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"? behaviors.tsv not found in {path}")
    
    df = pd.read_csv(
        file_path,
        sep="	",
        header=None,
        names=[
            "impression_id",
            "user_id",
            "time",
            "history",
            "impressions"
        ]
    )
    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    
    return df


train_behaviors = load_behaviors(TRAIN_PATH)
valid_behaviors = load_behaviors(VALID_PATH)

print("? Train behaviors:", train_behaviors.shape)
print("? Valid behaviors:", valid_behaviors.shape)

train_behaviors.head()

? Train behaviors: (156965, 5)
? Valid behaviors: (73152, 5)


,impression_id,user_id,time,history,impressions
0,1,U13740,2019-11-11 09:05:58,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
1,2,U91836,2019-11-12 18:11:30,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
2,3,U73700,2019-11-14 07:01:48,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...
3,4,U34670,2019-11-11 05:28:05,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0
4,5,U8125,2019-11-12 16:11:21,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...


In [18]:
# Load news.tsv

def load_news(path):
    file_path = os.path.join(path, "news.tsv")
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ news.tsv not found in {path}")
    
    df = pd.read_csv(
        file_path,
        sep="\t",
        header=None,
        names=[
            "news_id",
            "category",
            "subcategory",
            "title",
            "abstract",
            "url",
            "title_entities",
            "abstract_entities"
        ]
    )
    
    return df


train_news = load_news(TRAIN_PATH)
valid_news = load_news(VALID_PATH)

print("✅ Train news:", train_news.shape)
print("✅ Valid news:", valid_news.shape)

train_news.head()

✅ Train news: (51282, 8)
✅ Valid news: (42416, 8)


,news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."


In [19]:
# Build user click history from training behaviors


def parse_impressions(impressions):
    parsed = []
    for idx, token in enumerate(str(impressions).split()):
        news_id, label = token.rsplit('-', 1)
        parsed.append((idx, news_id, int(label)))
    return parsed


def dedupe_keep_last(items):
    seen = set()
    output = []
    for item in reversed(items):
        if item not in seen:
            seen.add(item)
            output.append(item)
    output.reverse()
    return output


def build_user_click_dict(df, max_history=50):
    user_clicks = defaultdict(list)
    sorted_df = df.sort_values(["user_id", "time", "impression_id"])

    for row in sorted_df.itertuples(index=False):
        if isinstance(row.history, str):
            user_clicks[row.user_id].extend(row.history.split())

        for _, news_id, label in parse_impressions(row.impressions):
            if label == 1:
                user_clicks[row.user_id].append(news_id)

    user_click_dict = {
        user_id: dedupe_keep_last(clicks)[-max_history:]
        for user_id, clicks in user_clicks.items()
        if clicks
    }
    return user_click_dict


user_click_dict = build_user_click_dict(train_behaviors)

print("Number of users:", len(user_click_dict))
example_user = next(iter(user_click_dict))
print("Example user:", example_user)
print("History sample:", user_click_dict[example_user][:5])


Number of users: 50000
Example user: U100
History sample: ['N20121', 'N33998', 'N45954', 'N55743', 'N50095']


## 3. Build item-item similarity matrix

This section implements the core ItemCF retrieval logic required by the project.

How it works:
- Each user's clicked news sequence contributes co-click evidence.
- Two news items become related if they appear in the same user history.
- The co-click signal is converted into an item-item similarity score.

Supported similarity formulas:
- **cosine**: `c_ij / sqrt(cnt(i) * cnt(j))`
- **jaccard**: `c_ij / (cnt(i) + cnt(j) - c_ij)`

Additional ItemCF improvements used in this notebook:
- session-length penalty to reduce noisy long histories,
- optional popularity penalty to reduce overly popular news,
- top-K neighbor truncation to remove weak relations,
- recency-aware scoring when ranking impression candidates.

These changes keep the method inside pure ItemCF while making the baseline stronger.


In [20]:
DEFAULT_ITEMCF_CONFIG = {
    "name": "cosine_recency_strong",
    "similarity_method": "cosine",
    "topk_neighbors": 150,
    "max_neighbors": 400,
    "max_history": 50,
    "recency_decay": 0.85,
    "neighbor_decay": 0.97,
    "use_popularity_penalty": True,
    "popularity_alpha": 0.50,
    "score_weights": {
        "sum": 1.00,
        "max": 0.10,
        "count": 0.06,
    },
}


def compute_similarity(cij, count_i, count_j, method):
    if method == "cosine":
        return cij / math.sqrt(count_i * count_j)
    if method == "jaccard":
        denom = count_i + count_j - cij
        return cij / denom if denom > 0 else 0.0
    raise ValueError(f"Unsupported similarity method: {method}")


def build_item_similarity(
    user_click_dict,
    max_history=50,
    max_neighbors=400,
    topk=150,
    similarity_method="cosine",
    use_popularity_penalty=True,
    popularity_alpha=0.50,
):
    print("Step 1: Building co-occurrence matrix...")

    item_count = defaultdict(int)
    co_occurrence = defaultdict(lambda: defaultdict(float))

    for _, items in tqdm(user_click_dict.items()):
        history = items[-max_history:]
        unique_items = list(dict.fromkeys(history))
        if len(unique_items) < 2:
            continue

        session_weight = 1.0 / math.log(1 + len(unique_items))

        for item in unique_items:
            item_count[item] += 1

        for i, item_i in enumerate(unique_items):
            for j, item_j in enumerate(unique_items):
                if i == j:
                    continue
                if len(co_occurrence[item_i]) < max_neighbors or item_j in co_occurrence[item_i]:
                    co_occurrence[item_i][item_j] += session_weight

    print("Step 2: Computing similarities...")
    item_sim = {}

    for item_i, related_items in tqdm(co_occurrence.items()):
        scored_neighbors = []
        count_i = item_count[item_i]

        for item_j, cij in related_items.items():
            count_j = item_count[item_j]
            sim = compute_similarity(cij, count_i, count_j, similarity_method)

            if use_popularity_penalty:
                sim /= math.pow(math.log(2 + count_j), popularity_alpha)

            scored_neighbors.append((item_j, sim))

        scored_neighbors.sort(key=lambda x: x[1], reverse=True)
        item_sim[item_i] = dict(scored_neighbors[:topk])

    print("Step 3: Keeping top-K similar items...")
    print(f"Similarity method: {similarity_method}")
    print(f"Top-K neighbors: {topk}")
    print("Item similarity ready!")
    return item_sim, item_count


In [21]:
def build_item_similarity_with_count(user_click_dict, config):
    return build_item_similarity(
        user_click_dict,
        max_history=config["max_history"],
        max_neighbors=config["max_neighbors"],
        topk=config["topk_neighbors"],
        similarity_method=config["similarity_method"],
        use_popularity_penalty=config["use_popularity_penalty"],
        popularity_alpha=config["popularity_alpha"],
    )


## 4. Define stronger ItemCF scoring settings

Instead of changing the model family, this notebook improves ItemCF by tuning the scoring rule itself.

Main scoring ideas:
- **recency decay**: more recent clicks should matter more,
- **neighbor decay**: higher-ranked neighbors should matter more,
- **sum similarity**: the main ItemCF evidence,
- **max similarity**: keeps a strong one-to-one match signal,
- **match count**: rewards candidates supported by multiple history items.

The final score is still an ItemCF score, because every component is derived from item-item similarity and user click history.


In [22]:
ITEMCF_CONFIGS = [
    DEFAULT_ITEMCF_CONFIG,
    {
        "name": "cosine_short_history",
        "similarity_method": "cosine",
        "topk_neighbors": 100,
        "max_neighbors": 300,
        "max_history": 30,
        "recency_decay": 0.88,
        "neighbor_decay": 0.97,
        "use_popularity_penalty": True,
        "popularity_alpha": 0.50,
        "score_weights": {"sum": 1.00, "max": 0.08, "count": 0.05},
    },
    {
        "name": "cosine_long_history",
        "similarity_method": "cosine",
        "topk_neighbors": 200,
        "max_neighbors": 500,
        "max_history": 80,
        "recency_decay": 0.82,
        "neighbor_decay": 0.96,
        "use_popularity_penalty": True,
        "popularity_alpha": 0.55,
        "score_weights": {"sum": 1.00, "max": 0.12, "count": 0.06},
    },
    {
        "name": "jaccard_balanced",
        "similarity_method": "jaccard",
        "topk_neighbors": 150,
        "max_neighbors": 400,
        "max_history": 50,
        "recency_decay": 0.85,
        "neighbor_decay": 0.97,
        "use_popularity_penalty": True,
        "popularity_alpha": 0.40,
        "score_weights": {"sum": 1.00, "max": 0.10, "count": 0.05},
    },
]


## 5. Generate ItemCF candidate scores

For each impression, the candidate set is already given by the MIND impression list.
The task of ItemCF is to assign a better score to each candidate.

Scoring pipeline:
- read the user's recent clicked history,
- fetch top-K neighbors for every history item,
- aggregate similarity using recency and neighbor-rank decay,
- combine sum-similarity, max-similarity, and support-count signals,
- fall back to a popularity prior when a candidate has no ItemCF support.


In [23]:
def generate_candidates_df(behaviors_df, item_sim, item_count, config):
    rows = []
    max_history = config["max_history"]
    recency_decay = config["recency_decay"]
    neighbor_decay = config["neighbor_decay"]
    popularity_alpha = config["popularity_alpha"]
    score_weights = config["score_weights"]

    for row in tqdm(behaviors_df.itertuples(index=False), total=len(behaviors_df), desc="Generating ItemCF scores"):
        history = row.history.split()[-max_history:] if isinstance(row.history, str) else []
        history_len = len(history)

        score_sum = defaultdict(float)
        score_max = defaultdict(float)
        support_count = defaultdict(int)

        for hist_idx, item in enumerate(reversed(history)):
            recency_weight = recency_decay ** hist_idx
            for neigh_rank, (related_item, sim_score) in enumerate(item_sim.get(item, {}).items()):
                neighbor_weight = neighbor_decay ** neigh_rank
                contribution = sim_score * recency_weight * neighbor_weight
                score_sum[related_item] += contribution
                score_max[related_item] = max(score_max[related_item], contribution)
                support_count[related_item] += 1

        if history_len > 0:
            norm = math.sqrt(history_len)
            for key in list(score_sum.keys()):
                score_sum[key] /= norm
                score_max[key] /= norm

        for candidate_index, news_id, label in parse_impressions(row.impressions):
            if news_id in score_sum:
                final_score = (
                    score_weights["sum"] * math.log1p(score_sum[news_id]) +
                    score_weights["max"] * score_max[news_id] +
                    score_weights["count"] * math.log1p(support_count[news_id])
                )
            else:
                final_score = 1.0 / math.pow(math.log(2 + item_count.get(news_id, 1)), popularity_alpha)

            rows.append(
                {
                    "impression_id": int(row.impression_id),
                    "candidate_index": int(candidate_index),
                    "news_id": news_id,
                    "label": int(label),
                    "score": float(final_score),
                }
            )

    return pd.DataFrame(rows)


## 6. Evaluate and save ItemCF outputs

This section contains shared evaluation and export utilities for the tuned ItemCF pipeline.

Saved outputs:
- scored validation table,
- ranking prediction file,
- metrics file,
- run log.

All exported results come from pure ItemCF only.


In [24]:
import datetime
import json
from sklearn.metrics import roc_auc_score


def compute_group_auc(df, score_col):
    aucs = []
    for _, group in df.groupby("impression_id", sort=False):
        if group["label"].nunique() < 2:
            continue
        aucs.append(roc_auc_score(group["label"], group[score_col]))
    return float(np.mean(aucs)) if aucs else float("nan")


def compute_mrr(df, score_col):
    values = []
    for _, group in df.groupby("impression_id", sort=False):
        ranked = group.sort_values(score_col, ascending=False)
        for rank, label in enumerate(ranked["label"].tolist(), start=1):
            if label == 1:
                values.append(1.0 / rank)
                break
    return float(np.mean(values)) if values else float("nan")


def dcg(labels):
    return sum((2 ** label - 1) / np.log2(idx + 2) for idx, label in enumerate(labels))


def compute_ndcg(df, score_col, k):
    scores = []
    for _, group in df.groupby("impression_id", sort=False):
        ranked = group.sort_values(score_col, ascending=False)
        labels = ranked["label"].tolist()[:k]
        ideal = sorted(labels, reverse=True)
        if not ideal or sum(ideal) == 0:
            continue
        scores.append(dcg(labels) / dcg(ideal))
    return float(np.mean(scores)) if scores else float("nan")


def collect_metrics(df, score_col, method_name):
    return {
        "method": method_name,
        "AUC": float(roc_auc_score(df["label"], df[score_col])),
        "GroupAUC": compute_group_auc(df, score_col),
        "MRR": compute_mrr(df, score_col),
        "nDCG@5": compute_ndcg(df, score_col, 5),
        "nDCG@10": compute_ndcg(df, score_col, 10),
    }


def save_rank_prediction(df, score_col, output_path):
    lines = []
    for impression_id, group in df.groupby("impression_id", sort=False):
        ordered = group.sort_values("candidate_index").copy()
        ordered["rank"] = ordered[score_col].rank(method="first", ascending=False).astype(int)
        lines.append(f"{impression_id} [{' '.join(map(str, ordered['rank'].tolist()))}]")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))


def save_metrics_json(metrics, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)


def save_log_file(metrics, output_path, config_name):
    log = (
        f"=== ItemCF Run Log ===\n"
        f"Time: {datetime.datetime.now()}\n"
        f"Config: {config_name}\n"
        f"AUC: {metrics['AUC']}\n"
        f"GroupAUC: {metrics['GroupAUC']}\n"
        f"MRR: {metrics['MRR']}\n"
        f"nDCG@5: {metrics['nDCG@5']}\n"
        f"nDCG@10: {metrics['nDCG@10']}\n"
    )
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(log)


In [25]:
def run_itemcf_pipeline(train_behaviors, valid_behaviors, config, save_outputs=False):
    print(f"Running config: {config['name']}")

    user_click_dict = build_user_click_dict(train_behaviors, max_history=config["max_history"])
    item_sim, item_count = build_item_similarity_with_count(user_click_dict, config)
    valid_df = generate_candidates_df(valid_behaviors, item_sim, item_count, config)

    metrics = collect_metrics(valid_df, score_col="score", method_name="itemcf")
    metrics["config_name"] = config["name"]
    metrics["similarity_method"] = config["similarity_method"]
    metrics["topk_neighbors"] = config["topk_neighbors"]
    metrics["max_history"] = config["max_history"]

    if save_outputs:
        output_dir = "outputs/itemcf"
        os.makedirs(output_dir, exist_ok=True)
        valid_df.to_csv(os.path.join(output_dir, "valid_scored.csv"), index=False)
        valid_df.to_parquet(os.path.join(output_dir, "valid_scored.parquet"), index=False)
        save_metrics_json(metrics, os.path.join(output_dir, "metrics.json"))
        save_rank_prediction(valid_df, "score", os.path.join(output_dir, "prediction.txt"))
        save_log_file(metrics, os.path.join(output_dir, "run.log"), config_name=config["name"])

    return valid_df, item_sim, item_count, metrics


## 7. Tune several ItemCF configurations

The notebook now compares several **ItemCF-only** settings.

What changes across configurations:
- similarity formula,
- top-K neighbors,
- history truncation,
- recency decay,
- neighbor decay,
- popularity penalty strength.

What stays fixed:
- the model family remains ItemCF,
- the ranking is always driven by item-item similarity from click co-occurrence.


In [26]:
tuning_results = []

for config in ITEMCF_CONFIGS:
    _, _, _, metrics = run_itemcf_pipeline(
        train_behaviors,
        valid_behaviors,
        config,
        save_outputs=False,
    )
    tuning_results.append(metrics)

tuning_results_df = pd.DataFrame(tuning_results).sort_values(
    ["nDCG@10", "MRR", "AUC"],
    ascending=False,
).reset_index(drop=True)

tuning_results_df


Running config: cosine_recency_strong
Step 1: Building co-occurrence matrix...


100%|██████████| 50000/50000 [00:06<00:00, 7185.44it/s]


Step 2: Computing similarities...


100%|██████████| 36784/36784 [00:03<00:00, 11388.22it/s]


Step 3: Keeping top-K similar items...
Similarity method: cosine
Top-K neighbors: 150
Item similarity ready!


...

Step 2: Computing similarities...


100%|██████████| 36784/36784 [00:03<00:00, 11377.67it/s]


Step 3: Keeping top-K similar items...
Similarity method: jaccard
Top-K neighbors: 150
Item similarity ready!


Generating ItemCF scores: 100%|██████████| 73152/73152 [02:05<00:00, 581.48it/s]


,method,AUC,GroupAUC,MRR,nDCG@5,nDCG@10,config_name,similarity_method,topk_neighbors,max_history
0,itemcf,0.501882,0.488727,0.237004,0.617597,0.519685,jaccard_balanced,jaccard,150,50
1,itemcf,0.501591,0.488649,0.236982,0.617437,0.519582,cosine_recency_strong,cosine,150,50
2,itemcf,0.501023,0.488085,0.237234,0.619226,0.519487,cosine_short_history,cosine,100,30
3,itemcf,0.502310,0.488845,0.236979,0.617664,0.519300,cosine_long_history,cosine,200,80


In [27]:
best_config_name = tuning_results_df.loc[0, "config_name"]
best_config = next(config for config in ITEMCF_CONFIGS if config["name"] == best_config_name)

print("Best ItemCF config:")
print(json.dumps(best_config, indent=2))

best_valid_df, best_item_sim, best_item_count, best_itemcf_metrics = run_itemcf_pipeline(
    train_behaviors,
    valid_behaviors,
    best_config,
    save_outputs=True,
)


Best ItemCF config:
{
  "name": "jaccard_balanced",
  "similarity_method": "jaccard",
  "topk_neighbors": 150,
  "max_neighbors": 400,
  "max_history": 50,
  "recency_decay": 0.85,
  "neighbor_decay": 0.97,
  "use_popularity_penalty": true,
  "popularity_alpha": 0.4,
  "score_weights": {
    "sum": 1.0,
    "max": 0.1,
    "count": 0.05
  }
}
Running config: jaccard_balanced
Step 1: Building co-occurrence matrix...


100%|██████████| 50000/50000 [00:06<00:00, 7369.39it/s]


Step 2: Computing similarities...


100%|██████████| 36784/36784 [00:03<00:00, 10172.25it/s]


Step 3: Keeping top-K similar items...
Similarity method: jaccard
Top-K neighbors: 150
Item similarity ready!


Generating ItemCF scores: 100%|██████████| 73152/73152 [02:07<00:00, 571.78it/s]


## 8. Select and export the best ItemCF model

This final section keeps only the best-performing pure ItemCF configuration and exports its outputs.

Use these files for the Member C submission:
- `outputs/itemcf/metrics.json`
- `outputs/itemcf/prediction.txt`
- `outputs/itemcf/run.log`
- `outputs/itemcf/valid_scored.csv`


In [28]:
best_itemcf_metrics

{'method': 'itemcf',
 'AUC': 0.5018824077510435,
 'GroupAUC': 0.48872668394270763,
 'MRR': 0.23700409600009975,
 'nDCG@5': 0.6175969395815026,
 'nDCG@10': 0.5196849177378744,
 'config_name': 'jaccard_balanced',
 'similarity_method': 'jaccard',
 'topk_neighbors': 150,
 'max_history': 50}

In [29]:
print("Saved ItemCF files in: outputs/itemcf")
print(f"Best config: {best_config['name']}")
print(f"Similarity method: {best_config['similarity_method']}")
print(f"Top-K neighbors: {best_config['topk_neighbors']}")
print(f"Max history: {best_config['max_history']}")

Saved ItemCF files in: outputs/itemcf
Best config: jaccard_balanced
Similarity method: jaccard
Top-K neighbors: 150
Max history: 50
